In [ ]:
!pip install opencv-python ultralytics

In [14]:
!pip install scikit-learn

   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------  8.7/8.7 MB 59.9 MB/s eta 0:00:01
   ---------------------------------------- 8.7/8.7 MB 45.3 MB/s  0:00:00

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
  

In [ ]:
# realtime_yolo.py
import cv2, time
from ultralytics import YOLO

model = YOLO("faceDetecter.pt")

cap = cv2.VideoCapture(0)  # 외장캠이면 1,2로 바꿔보기
if not cap.isOpened():
    raise RuntimeError("웹캠 못 염. 카메라 인덱스 확인 ㄱ")

prev = time.time()
fps = 0.0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    # 실시간 추론
    results = model(frame, imgsz=480, conf=0.8)[0]  # conf 조절 가능
    annotated = results.plot()  # 바운딩박스 그려진 프레임

    # FPS 계산
    now = time.time()
    fps = 0.9*fps + 0.1*(1.0/(now - prev))
    prev = now
    cv2.putText(annotated, f"FPS: {fps:.1f}", (10,30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("YOLO Realtime", annotated)
    if cv2.waitKey(1) & 0xFF == ord('q'):  # q 누르면 종료
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2, time, numpy as np
from ultralytics import YOLO

model = YOLO("./models/faceDetecter.pt")
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("딴카메라")

prev = time.time()
fps = 0.0
crop_id = 0 

def clamp_box(x1, y1, x2, y2, W, H):
    x1 = max(0, min(int(x1), W-1))
    y1 = max(0, min(int(y1), H-1))
    x2 = max(0, min(int(x2), W-1))
    y2 = max(0, min(int(y2), H-1))
    return x1, y1, x2, y2

while True:
    ok, frame = cap.read()
    if not ok:
        break
    H, W = frame.shape[:2]
    results = model(frame, imgsz=256, conf=0.7)[0]
    annotated = results.plot()
    if results.boxes is not None and len(results.boxes) > 0:
        boxes = results.boxes.xyxy.cpu().numpy()
        confs = results.boxes.conf.cpu().numpy()

        for i, (b, cf) in enumerate(zip(boxes, confs)):
            x1, y1, x2, y2 = clamp_box(*b, W, H)
            if x2 <= x1 or y2 <= y1:
                continue
            face = frame[y1:y2, x1:x2]
            if face.size == 0:
                continue

            face_small = cv2.resize(face, (160, 160), interpolation=cv2.INTER_LINEAR)
            cv2.imshow(f"face{i}", face_small)

            # cv2.imwrite(f"face_{crop_id:06d}.jpg", face)
            # crop_id += 1 파일로저장

        # areas = (boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1])
        # j = int(np.argmax(areas))
        # x1, y1, x2, y2 = clamp_box(*boxes[j], W, H)
        # main_face = frame[y1:y2, x1:x2]
        # cv2.imshow("main_face", cv2.resize(main_face, (160,160))) 가장 큰 얼굻만

    # FPS 표시
    now = time.time()
    fps = 0.9*fps + 0.1*(1.0/(now - prev))
    prev = now
    cv2.putText(annotated, f"FPS: {fps:.1f}", (10,30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("YOLO Realtime", annotated)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### 데이터분할

In [16]:
import os, shutil
from sklearn.model_selection import train_test_split
from pathlib import Path

# 원본 이미지가 있는 폴더
src_dir = Path("datasets\Drowsy")
all_imgs = list(src_dir.glob("*.png"))  # png면 확장자 맞춰
print(len(all_imgs), "개의 이미지 발견")

# train : val : test = 8 : 1 : 1
train_files, temp_files = train_test_split(all_imgs, test_size=0.2, random_state=42)
val_files, test_files  = train_test_split(temp_files, test_size=0.5, random_state=42)

# 저장할 폴더 생성
for split in ["train", "val", "test"]:
    (Path("dataset_split")/split).mkdir(parents=True, exist_ok=True)

def copy_files(files, dest):
    for f in files:
        shutil.copy(f, Path("dataset_split")/dest/f.name)

copy_files(train_files, "train")
copy_files(val_files,   "val")
copy_files(test_files,  "test")

print(f"Train:{len(train_files)}  Val:{len(val_files)}  Test:{len(test_files)}")


<>:6: SyntaxWarning: invalid escape sequence '\D'
<>:6: SyntaxWarning: invalid escape sequence '\D'
C:\Users\a8269\AppData\Local\Temp\ipykernel_24036\3381953371.py:6: SyntaxWarning: invalid escape sequence '\D'
  src_dir = Path("datasets\Drowsy")


22348 개의 이미지 발견
Train:17878  Val:2235  Test:2235


In [2]:
from ultralytics import YOLO

model = YOLO('yolo11n.yaml').load('yolo11n.pt')
model = YOLO('yolo11n.pt')
model.train(data = './data.yaml', epochs = 50, lr0= 0.01, lrf = 0.1, imgsz = 640, batch = 16, patience = 10 
            )

Transferred 499/499 items from pretrained weights
Ultralytics 8.3.199  Python-3.12.1 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce GTX 1660 SUPER, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True

c:\Users\a8269\OneDrive\바탕 화면\프로젝트\전국동아리\.venv\Lib\site-packages\ultralytics\utils\metrics.py:850: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
c:\Users\a8269\OneDrive\바탕 화면\프로젝트\전국동아리\.venv\Lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50       3.9G          0          0          0          0        640: 3% ──────────── 30/1118 3.7it/s 8.2s<4:55


KeyboardInterrupt: 

In [19]:
import torch
print(True if torch.cuda.is_available() else False)

False


In [1]:
import torch, sys
print("torch =", torch.__version__)
print("compiled cuda =", torch.version.cuda)
print("is_available =", torch.cuda.is_available())
print("num_devices =", torch.cuda.device_count())
print("current_device =", torch.cuda.current_device() if torch.cuda.is_available() else None)
print("name0 =", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("cudnn =", torch.backends.cudnn.version() if torch.cuda.is_available() else None)


torch = 2.6.0+cu124
compiled cuda = 12.4
is_available = True
num_devices = 1
current_device = 0
name0 = NVIDIA GeForce GTX 1660 SUPER
cudnn = 90100
